# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [3]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [4]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Thuvien = """SELECT dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat,
                        dbo.DecodeUTF8String(Thu_vien) AS Thu_vien, 
                        dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
                        gia_tri,
                        LocalLib
                        FROM Thu_vien
                        """
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)
print(df_thuvien)

                    Ten_viet_tat                            Thu_vien  \
0                         DHSPKT   Thư viện Đại học Sư Phạm Kĩ Thuật   
1                         ĐHSPKT                                None   
2                        SDHSPKT                                None   
3                    ĐHSPKT##Vie                                None   
4                            Vie                                None   
5                    DHSPKT##Vie                                None   
6                    D9HSP T.HCM                                None   
7                         SPDHKT                                None   
8                  ĐHSPKT TP.HCM                                None   
9                          ĐSPKT                                None   
10                        HCMUTE                                None   
11                           DLC                                None   
12  TVTTHCM|bvie|cTVTTHCM|eAACR2                                

C:\Users\admin\AppData\Local\Temp\ipykernel_11844\133294711.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_libol)


## Xử lý data

In [5]:
new_row = pd.DataFrame({'Ten_viet_tat': ['0'], # Tạo hàng dữ liệu giả lập cho thư viện không xác định
                        'Thu_vien': ['(Không xác định)'],
                        'Dia_chi': ['(Không xác định)'], 
                        'gia_tri': ['(Không xác định)'],
                        'LocalLib': ['False']})
df_thuvien = pd.concat([df_thuvien, new_row], ignore_index=True) # Thêm vào dataframe

df_thuvien = df_thuvien.sort_values(by="Ten_viet_tat", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_thuvien)

                    Ten_viet_tat                            Thu_vien  \
0                              0                    (Không xác định)   
1                            AU@                                None   
2                          AuCNL                                None   
3                             BC                                None   
4                          BTCTA                                None   
5                            CUD                                None   
6                    D9HSP T.HCM                                None   
7                        DDHSPKT                                None   
8                          DHSPK                                None   
9                         DHSPKT   Thư viện Đại học Sư Phạm Kĩ Thuật   
10                   DHSPKT##Vie                                None   
11                           DLC                                None   
12                      DNLM/DLC                                

## Load data

### [Nếu cần] Clear bảng

In [6]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Thu_vien"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [7]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Thu_vien (ID_thu_vien, Thu_vien, Dia_chi, Gia_tri, LocalLib) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_thuvien.iterrows():
    values = (row['Ten_viet_tat'], 
                row['Thu_vien'],
                row['Dia_chi'],
                row['gia_tri'],
                row['LocalLib'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()